# Unidad IV: clustering y PCA

## Objetivo

Agrupar sesiones por patrones de navegación sin usar `Revenue` para formar los grupos y reducir la dimensionalidad con PCA para visualizar los patrones. Se documenta el escalado, el criterio de selección de K, el tamaño de los grupos y la varianza explicada.

Se usan variables conductuales numéricas; los códigos de navegador/sistema/región/tráfico y la etiqueta de compra se excluyen del clustering. `Revenue` se consulta después para describir cada grupo, no participa en el ajuste.


In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

# Detectar raíz del proyecto de forma robusta
raiz = Path.cwd()
while raiz.name and raiz.name != 'REC_ECDB_2026_ChavezMolina_Byron':
    if raiz.parent == raiz:
        break
    raiz = raiz.parent

ruta = raiz / 'data' / 'processed' / 'clean' / 'online_shoppers_intention_clean.csv'
if not ruta.exists():
    raise FileNotFoundError('No se encontró el dataset procesado.')

datos = pd.read_csv(ruta)
variables_cluster = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
X = datos[variables_cluster]
escalador = RobustScaler()
X_escalado = escalador.fit_transform(X)
print(f'Sesiones analizadas: {len(X):,}')
print(f'Variables para agrupar: {len(variables_cluster)}')
print('Revenue se excluye del ajuste y se reserva para describir los clusters.')

## 1. Elegir una cantidad candidata de clusters

Se prueban valores de K entre 2 y 6. Se comparan la inercia (compactación interna, menor tiende a ser mejor) y el coeficiente silhouette (separación/cohesión, mayor tiende a ser mejor). El silhouette se calcula sobre una muestra reproducible de hasta 3,000 sesiones para controlar el costo de las distancias.


In [ ]:
{
  "cell_type": "code",
  "execution_count": 1,
  "id": "3ca3e075",
  "metadata": {},
  "outputs": [
    {
      "name": "stdout",
      "output_type": "stream",
      "text": [
        " K      Inercia  Silhouette_muestreado\n",
        " 2 8.475215e+07                 0.9343\n",
        " 3 4.190172e+07                 0.8910\n",
        " 4 2.564922e+07                 0.8700\n",
        " 5 1.754686e+07                 0.8425\n",
        " 6 1.395813e+07                 0.8109\n",
        "\n",
        "K candidata con mayor silhouette dentro del rango probado: 2\n"
      ]
    }
  ],
  "source": [
    "resultados_k = []\n",
    "for k in range(2, 7):\n",
    "    modelo_k = KMeans(n_clusters=k, n_init=10, random_state=42)\n",
    "    etiquetas_k = modelo_k.fit_predict(X_escalado)\n",
    "    silhouette = silhouette_score(\n",
    "        X_escalado, etiquetas_k,\n",
    "        sample_size=min(3000, len(X_escalado)), random_state=42\n",
    "    )\n",
    "    resultados_k.append({\n",
    "        'K': k,\n",
    "        'Inercia': modelo_k.inertia_,\n",
    "        'Silhouette_muestreado': silhouette,\n",
    "    })\n",
    "\n",
    "tabla_k = pd.DataFrame(resultados_k)\n",
    "print(tabla_k.round(4).to_string(index=False))\n",
    "mejor_k = int(tabla_k.sort_values('Silhouette_muestreado', ascending=False).iloc[0]['K'])\n",
    "print()\n",
    "print(f'K candidata con mayor silhouette dentro del rango probado: {mejor_k}')"
  ]
}

## 2. Ajustar K-Means y caracterizar los grupos

Se vuelve a ajustar K-Means con la K candidata sobre las 12,205 sesiones. La tabla de medianas permite describir los patrones de navegación; la tasa de `Revenue=True` ayuda a comparar los grupos después del ajuste, sin convertirla en entrada del algoritmo.


In [ ]:
{
  "cell_type": "code",
  "execution_count": 2,
  "id": "642a623e",
  "metadata": {},
  "outputs": [
    {
      "name": "stdout",
      "output_type": "stream",
      "text": [
        "         Sesiones  TasaCompra  Administrative_mediana  ProductRelated_mediana  DuracionProducto_mediana  PageValues_mediana\n",
        "Cluster                                                                                                                    \n",
        "0           11949       0.154                     1.0                    18.0                   591.167               0.000\n",
        "1             256       0.277                     5.0                    57.0                  2362.876               1.428\n",
        "\n",
        "Sesiones asignadas: 12205\n"
      ]
    }
  ],
  "source": [
    "modelo_kmeans = KMeans(n_clusters=mejor_k, n_init=20, random_state=42)\n",
    "clusters = modelo_kmeans.fit_predict(X_escalado)\n",
    "datos_cluster = datos.copy()\n",
    "datos_cluster['Cluster'] = clusters\n",
    "\n",
    "resumen_clusters = datos_cluster.groupby('Cluster').agg(\n",
    "    Sesiones=('Revenue', 'size'),\n",
    "    TasaCompra=('Revenue', 'mean'),\n",
    "    Administrative_mediana=('Administrative', 'median'),\n",
    "    ProductRelated_mediana=('ProductRelated', 'median'),\n",
    "    DuracionProducto_mediana=('ProductRelated_Duration', 'median'),\n",
    "    PageValues_mediana=('PageValues', 'median'),\n",
    ").round(3)\n",
    "print(resumen_clusters.to_string())\n",
    "print()\n",
    "print('Sesiones asignadas:', int(resumen_clusters['Sesiones'].sum()))\n",
    "\n",
    "carpeta_modelos = raiz / 'modelos'\n",
    "carpeta_modelos.mkdir(exist_ok=True)\n",
    "carpeta_datos = raiz / 'data' / 'processed' / 'clustering'\n",
    "carpeta_datos.mkdir(parents=True, exist_ok=True)\n",
    "joblib.dump(Pipeline([\n",
    "    ('escalador', RobustScaler()),\n",
    "    ('kmeans', KMeans(n_clusters=mejor_k, n_init=20, random_state=42)),\n",
    "]), carpeta_modelos / 'clustering' / 'kmeans_sesiones.joblib', compress=3)\n",
    "datos_cluster.to_csv(carpeta_datos / 'sesiones_clusterizadas.csv', index=False)\n",
    "tabla_k.to_csv(carpeta_modelos / 'metricas' / 'metricas_kmeans.csv', index=False)"
  ]
}

## 3. PCA para visualizar y resumir variación

PCA se ajusta sobre las mismas variables escaladas que K-Means. Se reporta cuánta varianza acumulan los dos primeros componentes. El plano de dos componentes sirve para visualizar; no se asume que preserve toda la estructura del espacio original.


In [ ]:
{
  "cell_type": "code",
  "execution_count": 3,
  "id": "fd871625",
  "metadata": {},
  "outputs": [
    {
      "name": "stdout",
      "output_type": "stream",
      "text": [
        "Varianza explicada por PC1: 98.204%\n",
        "Varianza explicada por PC2: 1.708%\n",
        "Varianza acumulada en 2 componentes: 99.912%\n"
      ]
    },
    {
      "data": {
        "image/png": "iVBORw0KGgoAAAANSUhEUgAAA0wAAAJOCAYAAACEIocIAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjExLjIsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvgI3uAAAAAAlwSFlzAAAPYQAAD2EBqD+naQAA/cJJREFUeJzs3Qd82+XVNuBbW957JXGcSXYCISGLEUbYM+xNGQUKlFFKSymFtpRReFtoC2W3rLJX2RtCCGFkkL0T23G8t7XX+zvHyLUdK7Ed25Kd+3o/fbaGpb8kh+r2Oc95DKFQKAQiIiIiIiLaiXHni4iIiIiIiIiBiYiIiIiIaBdYYSIiIiIiIoqAgYmIiIiIiCgCBiYiIiIiIqIIGJiIiIiIiIgiYGAiIiIiIiKKgIGJiIiIiIgoAgYmIiIiIiKiCBiYiIj2ck1NTRg3bhxOPvlkBAKBaB8OERFRTGFgIopBn3/+OQwGAz7++GMMRJs2bdLn9/jjj0f7UAa8zvwuXXLJJcjKysILL7wAk8nUp8dHkd1+++363rndbr5MRERRxMBEtBsbNmzAT37yE4waNQrx8fEYPnw4TjnlFLz99tsIBoN8/ahf++tf/4o1a9bgv//9L+x2e7QPp98577zzMGTIkGgfxoB04403amAMnywWCwoKCnD55ZejvLx8p9svXrwY5557rt5Gfpezs7Mxffp0DZ4lJSUdPsb111+v93344Yf3wTMiov6KgYloF3744QdMnTpVQ9Ozzz6LqqoqfPLJJ8jPz8eJJ56IRYsW9crrN3fuXIRCIRxxxBF8f6hXf5fkA+PKlSuRmprKV5pi0tatW/V3uK6uDn/5y1/w/PPP45BDDmlTefvjH/+I2bNnIy4uDm+++abedu3atRq65PYXXnjhTvfr9XrxzDPPaOD97LPPtPJNRNQRBiaiXXjggQfgcDjw4osvYubMmVphGjFiBP72t7/h1Vdfhc1m4+tHRNQHEhIScOqpp+K6665CfX19y23kr6I333yztvjJX0PlthdffDEKCwt3u+5k27Zt2gIkrSYS1vbZZx/84Q9/gM/na7nNG2+8oT8r7SnSbjV06FD98HDwwQdjxYoVOz2PnjrmSKTd6+ijj9awKa/jr371q4iDBTpzLB3pynNet25dmzYfud3EiRNx3333tbRYrl69Wq+Ty9qT52w0GvUv2l097lWrVuGoo45q81rIB7/267k6c4ytH1v+ii7vjdxOvsp71djY2Oe/S5Hel6+//hp33HEHBg8erL8/hx56KJYuXRrxtn/+858xbNgwfZ3lOIW0DM6ZM0c/NCcmJuoxfPjhhy0/f+utt+q/1Y5+J+W9kvvavHlzy2XfffedVonT09P1+U+aNAn/+te/dvpZuZ0Mw5A1XvLYM2bMwCuvvKLXHXjggXjuuee03av1+yVDNMRvf/vblsvk8aWKJx/cv\n...\n      (truncated - full base64 PNG data available in original notebook)\n"
      ],
      "output_type": "display_data"
    }
  ],
  "source": [
    "pca_2d = PCA(n_components=2, random_state=42)\n",
    "coordenadas = pca_2d.fit_transform(X_escalado)\n",
    "varianza = pca_2d.explained_variance_ratio_\n",
    "print(f'Varianza explicada por PC1: {varianza[0]:.3%}')\n",
    "print(f'Varianza explicada por PC2: {varianza[1]:.3%}')\n",
    "print(f'Varianza acumulada en 2 componentes: {varianza.sum():.3%}')\n",
    "\n",
    "carpeta_docs = raiz / 'docs' / 'evidencias'\n",
    "carpeta_docs.mkdir(exist_ok=True)\n",
    "fig, ax = plt.subplots(figsize=(9, 6))\n",
    "scatter = ax.scatter(\n",
    "    coordenadas[:, 0], coordenadas[:, 1], c=clusters,\n",
    "    cmap='tab10', s=12, alpha=0.55\n",
    ")\n",
    "ax.set_title('Sesiones de navegación proyectadas con PCA')\n",
    "ax.set_xlabel(f'PC1 ({varianza[0]:.1%} de varianza)')\n",
    "ax.set_ylabel(f'PC2 ({varianza[1]:.1%} de varianza)')\n",
    "fig.colorbar(scatter, ax=ax, label='Cluster K-Means')\n",
    "fig.tight_layout()\n",
    "fig.savefig(carpeta_docs / 'pca_clusters.png', dpi=160, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "joblib.dump(pca_2d, carpeta_modelos / 'clustering' / 'pca_sesiones.joblib', compress=3)\n",
    "carpeta_pca = raiz / 'data' / 'processed' / 'pca'\n",
    "carpeta_pca.mkdir(parents=True, exist_ok=True)\n",
    "pd.DataFrame({\n",
    "    'PC1': coordenadas[:, 0],\n",
    "    'PC2': coordenadas[:, 1],\n",
    "    'Cluster': clusters,\n",
    "    'Revenue': datos['Revenue'].astype(int),\n",
    "}).to_csv(carpeta_pca / 'coordenadas_pca.csv', index=False)\n",
    "pd.DataFrame({\n",
    "    'Componente': ['PC1', 'PC2'],\n",
    "    'VarianzaExplicada': varianza,\n",
    "    'VarianzaAcumulada': np.cumsum(varianza),\n",
    "}).to_csv(carpeta_modelos / 'metricas' / 'varianza_pca.csv', index=False)"
  ]
}

## Interpretación y límites

- Compara la silhouette, la inercia y el tamaño de los grupos; no existe una K universalmente correcta.
- Usa las medianas de navegación para describir el comportamiento de cada cluster y su tasa de compra como perfil posterior.
- PCA es una proyección: si los dos primeros componentes explican poca varianza, el gráfico bidimensional no representa bien todas las diferencias.
- Las categorías `Other` y las sesiones extremas pueden influir en los grupos. La estandarización robusta reduce la influencia de escalas y valores extremos, pero no los elimina.

El notebook guarda asignaciones, métricas y modelos en `data/processed/clustering/`, `data/processed/pca/`, `modelos/clustering/`, `modelos/metricas/`; la imagen se guarda en `docs/evidencias/pca_clusters.png`.